# Entity-swap relation 0: ours vs locally labeled k-means

This Colab notebook is intervention-only. Run k-means construction and LLM labeling locally first, then upload the prepared archive here. The notebook extracts two pre-labeled numeric summary dirs and one fixed 50-pair CSV, loads the replacement model, runs `eval_entity_swap.py` for `capital_country` relation 0, and compares the CSV results.

Local prep command from the repo root:

```bash
conda run -n circuit python -u eval/prepare_entity_swap_kmeans_artifacts.py \
  --ours-labeled-dir runs/label_summary_graphs/20260626_label_mntss_clt_gemma_2_2b_426k_entmax_alpha_0.50_node_0.02/labeled_output \
  --output-root runs/entity_swap_relation0_kmeans_artifacts \
  --sample-pairs 50 \
  --random-state 42 \
  --label-model gpt-4o-mini
```

Upload `runs/entity_swap_relation0_kmeans_artifacts.tar.gz` when prompted below.


In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found; CPU will be slow for entity-swap interventions.")


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/IamKrill1n/circuit_tracer_mod.git"
REPO_NAME = "circuit_tracer_mod"
REPO_REF = "clean_up"


def find_repo_root() -> Path | None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    candidates.extend([Path("/home/tu/circuit_tracer_mod"), Path("/content") / REPO_NAME])
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "summarization").is_dir():
            return candidate
    return None


REPO_ROOT = find_repo_root()
clone_parent = Path("/content") if Path("/content").exists() else Path.cwd()
if REPO_ROOT is None:
    subprocess.run(["git", "clone", REPO_URL, str(clone_parent / REPO_NAME)], check=True)
    REPO_ROOT = clone_parent / REPO_NAME

subprocess.run(["git", "fetch", "origin"], cwd=REPO_ROOT, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_ROOT, check=True)

print("repo root:", REPO_ROOT)
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, text=True).strip())


In [ ]:
from pathlib import Path
import subprocess
import sys

INSTALL_DEPS = Path("/content").exists()
if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], check=True)
else:
    print("Skipping dependency install outside Colab.")

help_text = subprocess.check_output(
    [sys.executable, "eval/eval_entity_swap.py", "--help"],
    cwd=REPO_ROOT,
    text=True,
)
assert "--pair-list" in help_text, "This checkout is missing the pair-list eval hook. Push/update the repo first."
print("entity-swap pair-list hook is available")


In [ ]:
import os
import shutil
from pathlib import Path

RELATION_IDX = 0
RELATION_NAME = "capital_country"
SAMPLE_PAIRS_PER_RELATION = 50
RANDOM_STATE = 42

PREP_ARCHIVE_PATH = Path("/content/entity_swap_relation0_kmeans_artifacts.tar.gz")
ARTIFACT_ROOT = REPO_ROOT / "colab_artifacts" / "entity_swap_relation0_kmeans"
OURS_EVAL_DIR = ARTIFACT_ROOT / "numeric_ours_labeled_relation0"
KMEANS_EVAL_DIR = ARTIFACT_ROOT / "numeric_kmeans_labeled_relation0"
PAIR_LIST_PATH = ARTIFACT_ROOT / "relation0_shared_pairs_sample50.csv"
OUTPUT_ROOT = ARTIFACT_ROOT / "entity_swap_outputs"
ANALOGIES_FILE = REPO_ROOT / "dataset" / "analogies" / "bats_analogies.txt"

MODEL_NAME = "google/gemma-2-2b"
TRANSCODER_SET = "mntss/clt-gemma-2-2b-2.5M"
BACKEND = "transformerlens"
DTYPE = "bfloat16"
DEVICE = "cuda" if shutil.which("nvidia-smi") else "cpu"

print("relation:", RELATION_IDX, RELATION_NAME)
print("expected pairs:", SAMPLE_PAIRS_PER_RELATION)
print("archive path:", PREP_ARCHIVE_PATH)
print("artifact root:", ARTIFACT_ROOT)
print("device:", DEVICE)


In [ ]:
import os

try:
    from google.colab import userdata
except Exception:
    userdata = None

# Optional: useful for gated model/tokenizer access and Hub rate limits. No LLM API key is needed here.
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_API_KEY")
if hf_token is None and userdata is not None:
    hf_token = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_API_KEY")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGINGFACE_API_KEY"] = hf_token
    print("Hugging Face token configured.")
else:
    print("No Hugging Face token configured. Continue only if model/tokenizer access works without it.")


In [ ]:
if not PREP_ARCHIVE_PATH.exists():
    try:
        from google.colab import files

        print("Upload entity_swap_relation0_kmeans_artifacts.tar.gz from local prep.")
        uploaded = files.upload()
        for name in uploaded:
            candidate = Path("/content") / name
            if name.endswith(".tar.gz"):
                PREP_ARCHIVE_PATH = candidate
                break
    except Exception as exc:
        raise FileNotFoundError(f"Archive not found: {PREP_ARCHIVE_PATH}") from exc

assert PREP_ARCHIVE_PATH.exists(), f"missing prepared archive: {PREP_ARCHIVE_PATH}"
print("using archive:", PREP_ARCHIVE_PATH)


In [ ]:
import argparse
import csv
import tarfile

import pandas as pd
import torch

from circuit_tracer import ReplacementModel
from eval.eval_entity_swap import DTYPE_MAP, run_entity_swap
from summarization.summarize import SummaryGraph

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
with tarfile.open(PREP_ARCHIVE_PATH, "r:gz") as tar:
    tar.extractall(ARTIFACT_ROOT)

print("extracted to:", ARTIFACT_ROOT)
print(sorted(path.name for path in ARTIFACT_ROOT.iterdir()))


In [ ]:
def numeric_count(path: Path) -> int:
    return len(list(path.glob("[0-9][0-9][0-9].sng.pt")))


def has_feature_labels(sng: SummaryGraph) -> bool:
    return any((sn.role or sn.description) for sn in sng.supernodes if sn.type not in ("emb", "logit"))


for method_dir in [OURS_EVAL_DIR, KMEANS_EVAL_DIR]:
    assert method_dir.exists(), f"missing {method_dir}"
    assert numeric_count(method_dir) == 100, f"expected 100 numeric summaries in {method_dir}"
    for idx in range(10):
        sng = SummaryGraph.load(str(method_dir / f"{idx:03d}.sng.pt"))
        assert has_feature_labels(sng), f"{method_dir}/{idx:03d}.sng.pt is not labeled"

assert PAIR_LIST_PATH.exists(), f"missing pair list: {PAIR_LIST_PATH}"
with PAIR_LIST_PATH.open(encoding="utf-8", newline="") as f:
    pair_rows = list(csv.DictReader(f))
assert len(pair_rows) == SAMPLE_PAIRS_PER_RELATION, len(pair_rows)
assert ANALOGIES_FILE.exists(), f"missing {ANALOGIES_FILE}"

print("ours numeric files:", numeric_count(OURS_EVAL_DIR))
print("k-means numeric files:", numeric_count(KMEANS_EVAL_DIR))
print("pair rows:", len(pair_rows))


In [ ]:
print("loading replacement model once")
model = ReplacementModel.from_pretrained(
    MODEL_NAME,
    TRANSCODER_SET,
    backend=BACKEND,
    lazy_encoder=True,
    dtype=DTYPE_MAP[DTYPE],
    device=torch.device(DEVICE) if DEVICE else None,
)


def run_method(method: str, graph_dir: Path) -> Path:
    output_dir = OUTPUT_ROOT / method
    args = argparse.Namespace(
        negation_coefficients="-2",
        addition_coefficients="2,4,8",
        relations=str(RELATION_IDX),
        graph_dir=graph_dir,
        analogies_file=ANALOGIES_FILE,
        output_dir=output_dir,
        layers_below=0,
        layers_above=1,
        sample_pairs_per_relation=None,
        random_state=RANDOM_STATE,
        pair_list=PAIR_LIST_PATH,
    )
    print(f"running entity swap: {method}", flush=True)
    run_entity_swap(model, args)
    return output_dir


ours_output = run_method("ours-ilp", OURS_EVAL_DIR)
kmeans_output = run_method("baseline-kmeans", KMEANS_EVAL_DIR)
print("outputs:", ours_output, kmeans_output)


In [ ]:
def summarize_method(method: str, output_dir: Path) -> tuple[pd.DataFrame, dict[str, float | int | str]]:
    results = pd.read_csv(output_dir / "swap_results.csv")
    summary = pd.read_csv(output_dir / "swap_summary.csv")
    pair_count = results[["source_idx", "donor_idx"]].drop_duplicates().shape[0]
    metrics = {
        "method": method,
        "pairs": int(pair_count),
        "rows": int(len(results)),
        "success": int(results["success"].sum()),
        "success_rate": float(results["success"].mean()) if len(results) else float("nan"),
        "success_exact": int(results["success_exact"].sum()),
        "top1_donor": int(results["top1_is_donor"].sum()),
        "top5_donor": int(results["top5_has_donor"].sum()),
        "mean_p_source_clean": float(results["p_source_clean"].mean()) if len(results) else float("nan"),
        "mean_p_source_steered": float(results["p_source_steered"].mean()) if len(results) else float("nan"),
        "mean_p_donor_clean": float(results["p_donor_clean"].mean()) if len(results) else float("nan"),
        "mean_p_donor_steered": float(results["p_donor_steered"].mean()) if len(results) else float("nan"),
    }
    summary.insert(0, "method", method)
    return summary, metrics


ours_summary, ours_metrics = summarize_method("ours-ilp", ours_output)
kmeans_summary, kmeans_metrics = summarize_method("baseline-kmeans", kmeans_output)
metrics_df = pd.DataFrame([ours_metrics, kmeans_metrics])
summary_df = pd.concat([ours_summary, kmeans_summary], ignore_index=True)

display(metrics_df)
display(summary_df[[
    "method",
    "relation_idx",
    "relation_name",
    "source_factor",
    "donor_factor",
    "n_attempted",
    "n_success",
    "success_rate",
    "n_top5_hits",
    "top5_hit_rate",
]])
print("pair list:", PAIR_LIST_PATH)
print("output root:", OUTPUT_ROOT)
